In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers
import os

import tensorflow_datasets as tfds
dataset_name = 'cats_vs_dogs'
dataset, info = tfds.load(name=dataset_name, with_info=True, as_supervised=True)

train_data = dataset["train"].take(20000) #take 20000 images for training
val_data = dataset["train"].skip(20000).take(5000) #skip the first 20000 images and take the
#  next 5000 images for validation

In [ ]:
#Preprocessing images
IMG_SIZE = 128  # Resize images to 128x128 pixels

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))  # Resize image
    image = image / 255.0  # Normalize pixel values
    return image, label

# Apply preprocessing to training and validation sets
train_data = train_data.map(preprocess).batch(32).shuffle(1000)
val_data = val_data.map(preprocess).batch(32)

In [ ]:

model = keras.Sequential([
    # First Convolutional Layer
    layers.Conv2D(32, (3, 3), activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(2, 2),

    # Second Convolutional Layer
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D(2, 2),

    # Third Convolutional Layer
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D(2, 2),

    # Flattening Layer
    layers.Flatten(),

    # Fully Connected Layer
    layers.Dense(512, activation="relu"),

    # Output Layer (Binary Classification)
    layers.Dense(1, activation="sigmoid")
])

# Compile the model
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# Display model summary
model.summary()


In [ ]:
#Step 5: Train the model
history = model.fit(train_data, epochs=10, validation_data=val_data)

In [ ]:
#Step 6: plot training performance
plt.figure(figsize=(12, 4))

#accuracy plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

#loss plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()


In [ ]:
# Step 1: Import required libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Step 2: Upload image (simpler and safer than hardcoding path)
from google.colab import files
uploaded = files.upload()

# Step 3: Get uploaded image name
img_path = list(uploaded.keys())[0]

# Step 4: Load and preprocess the image
image = cv2.imread(img_path)                      # Read image
image = cv2.resize(image, (128, 128))             # Resize to match model input
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)    # Convert BGR to RGB
image = image / 255.0                             # Normalize pixel values (0–1)
image = np.expand_dims(image, axis=0)             # Add batch dimension

# Step 5: Show the image
plt.imshow(image[0])
plt.title("Preprocessed Image")
plt.axis("off")
plt.show()


In [ ]:
# Predict the class using the trained model
prediction = model.predict(image)

# prediction is an array like [[0.92]], so we take the first value
predicted_value = prediction[0][0]

# Print the probability
print(f"Model confidence: {predicted_value:.2f}")

# Make decision based on threshold (0.5)
if predicted_value > 0.5:
    print("The model predicts: Dog 🐶")
else:
    print("The model predicts: Cat 🐱")
